# Notebook 02: Infrastructure Timeline Audit & Leakage Protection

## Analytical Question
How can we construct historical panels of locality infrastructure exposure without introducing look-ahead leakage?

## Methodology
We run the temporal stage reconstruction function `get_project_stage_at_quarter` on a mock project timeline (Jewar Airport) to verify that stages change dynamically based strictly on dates.

In [ ]:
import sys
from datetime import date
sys.path.append("../")
from src.ncr_intelligence.features.infrastructure_features import get_project_stage_at_quarter

# Timeline for Jewar Airport (Upcoming Noida International Airport)
airport_events = [
    {"stage": "PROPOSED", "event_date": date(2016, 12, 1), "article_publish_date": date(2016, 12, 3)},
    {"stage": "APPROVED", "event_date": date(2018, 5, 10), "article_publish_date": date(2018, 5, 12)},
    {"stage": "CONTRACTED", "event_date": date(2020, 7, 31), "article_publish_date": date(2020, 8, 2)},
    {"stage": "UNDER_CONSTRUCTION", "event_date": date(2021, 8, 20), "article_publish_date": date(2021, 8, 22)}
]

# Check status at different historical quarters
test_quarters = ["2016-Q3", "2017-Q2", "2019-Q1", "2020-Q4", "2023-Q2"]
for qtr in test_quarters:
    stage = get_project_stage_at_quarter(airport_events, qtr)
    print(f"Project stage as of {qtr}: {stage}")

### Verification of Information Availability (Delayed Publication Test)

In [ ]:
# Demonstration where event date is Q3 but official publication is delayed to Q4
delayed_events = [
    {"stage": "UNDER_CONSTRUCTION", "event_date": date(2022, 8, 1), "article_publish_date": date(2022, 10, 15)}
]

status_q3 = get_project_stage_at_quarter(delayed_events, "2022-Q3")
status_q4 = get_project_stage_at_quarter(delayed_events, "2022-Q4")

print(f"Under construction status as of 2022-Q3 (before news publish): {status_q3}")
print(f"Under construction status as of 2022-Q4 (after news publish): {status_q4}")

## Findings & Limitations
- **Findings**: The temporal stage reconstructor correctly suppresses status changes if the publication date falls after the target quarter.
- **Limitations**: Preserving separate event dates and publication dates requires strict source metadata collation during crawler development.

## Decision
All ingestion adapters in later phases must capture publication timestamps alongside event dates to prevent leakage during model training.